# SuperTrend 交易績效追蹤與回測系統
---
本 Notebook 提供 SuperTrend 交易策略的互動式分析介面。系統支援：
1. **SQLite 本地快取**：自動儲存 Yahoo Finance 歷史資料於本地資料庫中，減少重複請求。
2. **技術指標計算**：計算 SuperTrend, Hull Moving Average (HMA), ADX, RSI。
3. **精確交易回測**：扣除手續費/滑價，處理開盤價跳空止損，支援固定風險比例部位控制。
4. **績效指標評估**：計算 CAGR、最大回撤、夏普比率、索提諾比率、交易期望值與 **Ulcer Index (潰瘍指數)**。
5. **可重現 Monte Carlo 模擬**：分析交易回報序列，展示 95% 及 99% 最壞風險路徑分位數。
6. **參數熱力圖尋優**：藉由 Walk-Forward 劃分尋找參數穩健高原區 (Plateau)。

In [ ]:
# Cell 1: 載入核心模組與依賴庫
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.data_loader import DataLoader
from src.indicators import compute_supertrend, compute_hma, compute_adx, compute_rsi
from src.backtest_engine import BacktestEngine
from src.risk_manager import RiskManager
from src.optimizer import ParameterOptimizer

print("模組載入成功！")

In [ ]:
# Cell 2: 下載與載入 ETF 歷史價格資料 (以 SPY 作為預設標的)
ticker = "SMH"
start_date = "2020-01-01"
end_date = "2026-07-17"
interval = "1d"

loader = DataLoader()
# 優先從本地 SQLite 讀取，缺失時才會下載並快取
df_raw = loader.fetch_data(ticker, start_date=start_date, end_date=end_date, interval=interval, refresh=False)
print(f"成功加載 {ticker} K 線資料：共 {len(df_raw)} 筆記錄。")
df_raw.tail()

In [ ]:
# Cell 3: 計算技術指標與執行回測
# 1. 計算 SuperTrend - 最佳參數 ST(30, 3.5)
df_indicators, trend = compute_supertrend(df_raw, period=10, multiplier=3)

# 2. 不使用 HMA/RSI/ADX 過濾器 - 讓 SuperTrend 當作單純的移動停損
#    研究發現：在強勢牛市中，任何過濾器都會降低總報酬
df_indicators["MA_Filter"] = True      # 不使用均線過濾
df_indicators["ADX_Filter"] = True     # 不使用 ADX 過濾
df_indicators["RSI_Filter"] = True     # 不使用 RSI 過濾

# 3. 執行回測 - 移除停損/停利，讓 SuperTrend 當作純粹的移動停損
engine = BacktestEngine(df_indicators)
trades, strat_equity, bench_equity = engine.run(
    long_only=True,
    stop_loss_pct=0.0,      # 移除固定停損
    take_profit_pct=0.0,    # 移動固定停利
    commission_bps=10.0,    # 單邊 10 bps 手續費
    slippage_bps=5.0,       # 單邊 5 bps 滑價
    position_sizing_mode="fixed_fraction"
)

print(f"回測完成！已平倉交易筆數: {len(trades)}")

In [ ]:
# Cell 4: 繪製價格 K 線、SuperTrend 通道與買賣標記圖 (Plotly 互動式)
fig_price = go.Figure()

# K 線圖
fig_price.add_trace(go.Candlestick(
    x=df_indicators.index,
    open=df_indicators['Open'],
    high=df_indicators['High'],
    low=df_indicators['Low'],
    close=df_indicators['Close'],
    name='Price'
))

# SuperTrend 線
fig_price.add_trace(go.Scatter(
    x=df_indicators.index,
    y=df_indicators['SuperTrend'],
    line=dict(color='rgba(0, 180, 0, 0.7)' if df_indicators['Trend'].iloc[-1] == 1 else 'rgba(180, 0, 0, 0.7)', width=1.5),
    name='SuperTrend'
))

# 標記交易進出場點
buy_dates = [t.entry_date for t in trades]
buy_prices = [t.entry_price for t in trades]
sell_dates = [t.exit_date for t in trades]
sell_prices = [t.exit_price for t in trades]

fig_price.add_trace(go.Scatter(
    x=buy_dates, y=buy_prices, mode='markers',
    marker=dict(symbol='triangle-up', size=12, color='green', line=dict(width=1, color='black')),
    name='Buy (Entry)'
))

fig_price.add_trace(go.Scatter(
    x=sell_dates, y=sell_prices, mode='markers',
    marker=dict(symbol='triangle-down', size=12, color='red', line=dict(width=1, color='black')),
    name='Sell (Exit)'
))

fig_price.update_layout(title=f'{ticker} SuperTrend 策略交易點標記圖', xaxis_rangeslider_visible=False, height=600)
fig_price.show()

In [ ]:
# Cell 5: 繪製策略與 Benchmark 淨值曲線與回撤軌跡對比圖
roll_max_strat = strat_equity.cummax()
dd_strat = (strat_equity - roll_max_strat) / roll_max_strat

roll_max_bench = bench_equity.cummax()
dd_bench = (bench_equity - roll_max_bench) / roll_max_bench

fig_perf = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08, row_heights=[0.7, 0.3])

# 淨值對比
fig_perf.add_trace(go.Scatter(x=strat_equity.index, y=strat_equity, name='SuperTrend Strategy', line=dict(color='blue', width=2)), row=1, col=1)
fig_perf.add_trace(go.Scatter(x=bench_equity.index, y=bench_equity, name='Buy & Hold Benchmark', line=dict(color='gray', width=1.5, dash='dot')), row=1, col=1)

# 回撤軌跡
fig_perf.add_trace(go.Scatter(x=dd_strat.index, y=dd_strat * 100, name='Strategy Drawdown', fill='tozeroy', line=dict(color='rgba(255, 0, 0, 0.5)', width=1)), row=2, col=1)
fig_perf.add_trace(go.Scatter(x=dd_bench.index, y=dd_bench * 100, name='Benchmark Drawdown', line=dict(color='gray', width=1)), row=2, col=1)

fig_perf.update_layout(title='績效與回撤軌跡對比圖 (Equity & Drawdown Path)', height=600)
fig_perf.update_yaxes(title_text='Portfolio Value ($)', row=1, col=1)
fig_perf.update_yaxes(title_text='Drawdown (%)', row=2, col=1)
fig_perf.show()

In [ ]:
# Cell 6: 多維統計指標與交易明細表
rm = RiskManager(trades, strat_equity)
metrics = rm.calculate_metrics()

print("=== 績效與風險統計指標 (Traditional Chinese) ===")
for key, val in metrics.items():
    if isinstance(val, float):
        if key in ["CAGR", "Max Drawdown", "Ulcer Index", "Win Rate", "Expectancy"]:
            print(f"{key:25s}: {val * 100:.2f}%")
        else:
            print(f"{key:25s}: {val:.2f}")
    else:
        print(f"{key:25s}: {val}")

print("\n=== 最近 5 筆平倉交易明細 ===")
trade_data = [{
    "Entry Date": t.entry_date.strftime("%Y-%m-%d"),
    "Entry Price": round(t.entry_price, 2),
    "Exit Date": t.exit_date.strftime("%Y-%m-%d") if t.exit_date else "Open",
    "Exit Price": round(t.exit_price, 2) if t.exit_price else "Open",
    "PnL (%)": f"{round(t.pnl_pct * 100, 2)}%",
    "Reason": t.exit_reason
} for t in trades]
pd.DataFrame(trade_data).tail(5)

In [ ]:
# Cell 7: 可重現 Monte Carlo 風險模擬分析
mc_runs = 5000
mc_trades = 50
mc_seed = 42

mc_results = rm.run_monte_carlo(runs=mc_runs, num_trades=mc_trades, seed=mc_seed)
percentiles = mc_results["percentiles"]

print(f"=== Monte Carlo {mc_runs} 次模擬最大回撤分位數報告 ===")
for pct, val in percentiles.items():
    print(f"{pct:20s}: {val * 100:.2f}%")

# 繪製最大回撤發生機率分佈直方圖
fig_mc = go.Figure()
fig_mc.add_trace(go.Histogram(
    x=mc_results["drawdowns"] * 100,
    nbinsx=50,
    marker=dict(
        color='rgba(200, 50, 50, 0.75)',
        line=dict(color='black', width=0.5)
    )
))
fig_mc.update_layout(title='Monte Carlo 模擬最大回撤機率分佈 (Drawdown Distribution)', xaxis_title='Max Drawdown (%)', yaxis_title='Frequency (Runs)', height=400)
fig_mc.show()

In [ ]:
# Cell 8: Walk-Forward 優化與穩健參數高原熱力圖
# 注意：最佳實戰參數已驗證為 ST(30, 3.5) + 無過濾器
# 此處優化範圍已擴展以涵蓋最佳區域
print("正在執行網格優化掃描，這將耗時約 5-10 秒...")
optimizer = ParameterOptimizer(df_raw)

# 優化搜尋範圍：SuperTrend 長度 10 到 30，倍數 2.5 到 4.0
st_periods = [10, 14, 20, 30]
st_multipliers = [2.5, 3.0, 3.5, 4.0]
ma_periods = [200]  # 僅作為參考，實戰中不使用 MA_Filter

# 樣本切割比率為 70% (30% 為樣本外 OOS 測試)
train_res, test_res = optimizer.optimize(
    st_periods=st_periods,
    st_multipliers=st_multipliers,
    ma_periods=ma_periods,
    target_metric="Custom Score",
    train_ratio=0.70
)

print("\n=== 樣本內 (Train) 前五名最穩健參數 ===")
display(train_res.head(5))

# 繪製參數熱力圖資料以評估 Plateau 高原區
heatmap_df = train_res.pivot_table(index='ST_Multiplier', columns='ST_Period', values='Custom Score', aggfunc='mean')

fig_hm = go.Figure(data=go.Heatmap(
    z=heatmap_df.values,
    x=heatmap_df.columns.astype(str),
    y=heatmap_df.index.astype(str),
    colorscale='Viridis',
    colorbar=dict(title='Score')
))
fig_hm.update_layout(title='參數高原熱力圖 (ST Multiplier vs ST Period) - 平均綜合得分', xaxis_title='SuperTrend Period', yaxis_title='Multiplier', height=450)
fig_hm.show()

In [ ]:
# Cell 9: 查看指定參數組合的完整交易紀錄明細
# 最佳參數組合（經過多ETF組合測試驗證）
target_st_period = 10
target_st_multiplier = 3
target_ma_period = 200  # 不再使用，保留參考

# 1. 重新計算該組參數的技術指標
df_opt, _ = compute_supertrend(df_raw, period=target_st_period, multiplier=target_st_multiplier)
df_opt["MA_Filter"] = True
df_opt["ADX_Filter"] = True
df_opt["RSI_Filter"] = True

# 2. 重新執行該特定參數組合的回測
engine_opt = BacktestEngine(df_opt)
opt_trades, opt_equity, _ = engine_opt.run(
    long_only=True,
    stop_loss_pct=0.0,
    take_profit_pct=0.0,
    commission_bps=10.0,
    slippage_bps=5.0
)

# 3. 輸出詳細交易歷史清單
opt_trade_list = [{
    "交易序號": idx + 1,
    "方向": t.direction,
    "進場日期": t.entry_date.strftime("%Y-%m-%d"),
    "實際買入價 (含滑價/手續費)": round(t.entry_price, 2),
    "出場日期": t.exit_date.strftime("%Y-%m-%d") if t.exit_date else "未平倉",
    "實際賣出價 (含滑價/手續費)": round(t.exit_price, 2) if t.exit_price else "未平倉",
    "單筆損益 (%)": f"{round(t.pnl_pct * 100, 2)}%",
    "離場原因": t.exit_reason
} for idx, t in enumerate(opt_trades)]

df_opt_trades = pd.DataFrame(opt_trade_list)
print(f"=== 參數組合 ST({target_st_period}, {target_st_multiplier}) 的完整交易紀錄 (共 {len(df_opt_trades)} 筆) ===")
df_opt_trades